# Motion-representation zoo — kinematic comparison

Compares all four motion representations natively supported by Flow Planner (Waypoints, Frenet, Velocity, Acceleration) at a paired single seed (seed=269) and 1,500 training scenarios. Produces the motion-matrix bar chart that appears in the paper.

Set the `model.kinematic=` flag to switch between representations. The trained checkpoints from this notebook supply the per-representation eval JSONs referenced by `cs269_generate_paper_bevs.ipynb`.

**Setup requirements**: see `QUICKSTART.md` at the repo root.

## 1. Config

**This is the only cell you edit per run.** Set `KINEMATIC` (`waypoints` or `frenet`), `SEED`, and the Drive folder. Everything else is derived. To force re-preprocessing, flip the optional `WIPE_*` flags at the bottom.

In [ ]:
# Edit only if you want non-default values
DRIVE_FOLDER         = '/content/drive/MyDrive/cs269'
DRIVE_OUT_MOTION     = f'{DRIVE_FOLDER}/motion_representations'
RUN_SEED             = 269   # v7: paired retrain at seed 269 (was 42 in v6)

# ===== Motion-zoo notebook: loops over 4 kinematics at one seed =====
KINEMATICS_TO_RUN = ['waypoints', 'velocity', 'acceleration', 'frenet']
SEEDS_TO_RUN      = [269]
RUN_FEASIBILITY   = True   # post-process bicycle-feasibility analysis on each variant
V7_VARIANT        = 'AUDIT'  # downstream filename tag

TOTAL_SCENARIOS      = 1500
assert TOTAL_SCENARIOS is None or TOTAL_SCENARIOS >= 1000, (
    f'TOTAL_SCENARIOS={TOTAL_SCENARIOS}: must be None (unlimited) or >= 1000. '
    'A cap below 1000 produces a degenerate training set. '
    'A value of 0 is ambiguous across nuPlan versions; use None instead.'
)

TRAIN_EPOCHS         = 50
TRAIN_BATCH_SIZE     = 32

# Held-out test set (best-effort - skipped if raw nuPlan data not available).
# After the Phase 2 methodology rewrite, log-disjoint train/val split is
# enforced via --log_names_json instead of an offset-based skip. The
# HELDOUT_SCENARIOS value here is a safety cap on the val-log preprocess;
# the actual val cache size depends on how many scenarios live in the val logs.
#
# Scale-up plan: when running the full mini-split (TOTAL_SCENARIOS=None),
# bump the held-out cap proportionally so the eval signal stays
# statistically meaningful. The val logs hold ~1500-2000 scenarios total.
HELDOUT_SCENARIOS    = 300 if (TOTAL_SCENARIOS is not None and TOTAL_SCENARIOS <= 2000) else 1500

WARM_UP_EPOCHS       = min(5, max(0, TRAIN_EPOCHS - 1))
EFFECTIVE_BATCH_SIZE = min(TRAIN_BATCH_SIZE, TOTAL_SCENARIOS) if TOTAL_SCENARIOS else TRAIN_BATCH_SIZE

assert TRAIN_EPOCHS == 0 or WARM_UP_EPOCHS < TRAIN_EPOCHS, (
    f'WARM_UP_EPOCHS ({WARM_UP_EPOCHS}) must be < TRAIN_EPOCHS ({TRAIN_EPOCHS})'
)

# Drive paths
DRIVE_ZIPS              = f'{DRIVE_FOLDER}/nuplan_zips'
DRIVE_CKPT              = f'{DRIVE_FOLDER}/checkpoints'
DRIVE_RESULTS           = f'{DRIVE_FOLDER}/results'
DRIVE_REQUIREMENTS      = f'{DRIVE_FOLDER}/working_requirements.txt'
# Scale-aware Drive cache folder: when running at full scale we point at a
# distinct Drive folder so the smaller 1500-scenario cache is not clobbered.
DRIVE_PREPROCESSED_1500 = f'{DRIVE_FOLDER}/preprocessed_cache_1500'
DRIVE_PREPROCESSED_FULL = f'{DRIVE_FOLDER}/preprocessed_cache_full'
DRIVE_PREPROCESSED      = DRIVE_PREPROCESSED_FULL if TOTAL_SCENARIOS is None else DRIVE_PREPROCESSED_1500
DRIVE_HELDOUT           = f'{DRIVE_FOLDER}/heldout_cache_{HELDOUT_SCENARIOS}'

# Local paths
LOCAL_ROOT           = '/content/work'
LOCAL_NUPLAN         = f'{LOCAL_ROOT}/nuplan'
LOCAL_MAPS           = f'{LOCAL_NUPLAN}/maps'
LOCAL_LOGS           = f'{LOCAL_NUPLAN}/data/cache/mini'
LOCAL_EXP            = f'{LOCAL_NUPLAN}/exp'
LOCAL_CACHE          = f'{LOCAL_ROOT}/preprocessed_cache'
LOCAL_HELDOUT_CACHE  = f'{LOCAL_ROOT}/heldout_cache'
LOCAL_RUNS           = f'{LOCAL_ROOT}/runs'
LOCAL_TB             = f'{LOCAL_ROOT}/tensorboard'

# Code / env paths
REPO_DIR             = '/content/CS269FlowPlannerProject'
FP_DIR               = f'{REPO_DIR}/flow_planner'
NUPLAN_DEVKIT_DIR    = '/content/nuplan-devkit'
VENV                 = '/content/venv39'
PYTHON               = f'{VENV}/bin/python'
PIP                  = f'{VENV}/bin/pip'
CONSTRAINTS          = f'{LOCAL_ROOT}/pip_constraints.txt'  # live under LOCAL_ROOT so /tmp wipe doesn't lose it

# Methodology: log-disjoint train/val split (Phase 2 deliverable).
# Split seed intentionally pinned to 42 so v6 / v7 train on the SAME logs -
# only RUN_SEED (model init) varies between runs. Do NOT roll this to 269.
# See scripts/generate_log_split.py and docs/preprocessing_methodology.md s4.
# If docs/log_split_mini_seed42.json is not committed to the repo, cells 31
# and 41 generate it lazily once LOCAL_LOGS is populated.
LOG_SPLIT_JSON       = f'{REPO_DIR}/docs/log_split_mini_seed42.json'

# Tracks whether held-out eval is available this session (set later)
HELDOUT_AVAILABLE = False

# Sentinel so later cells can assert this cell ran (e.g. after kernel restart)
_CONFIG_CELL_RAN = True

_total_label = 'unlimited (full mini-split)' if TOTAL_SCENARIOS is None else str(TOTAL_SCENARIOS)
print(f'Train scenarios cap: {_total_label}, held-out cap: {HELDOUT_SCENARIOS}')
print(f'Epochs: {TRAIN_EPOCHS} (warmup {WARM_UP_EPOCHS}), batch: {EFFECTIVE_BATCH_SIZE}')
print(f'Drive cache: {DRIVE_PREPROCESSED}')
print(f'Log split: {LOG_SPLIT_JSON} (seed 42 pinned; RUN_SEED={RUN_SEED} is model-init only)')

# ---- Optional: force re-preprocessing by uncommenting one or both flags ----
# Optional: wipe preprocessed caches before this run.
#
# WIPE_LOCAL_CACHE  is cheap — just /content/work; survives any later cell.
# WIPE_DRIVE_CACHE  is EXPENSIVE — destroys preprocessed_cache_1500 on Drive,
#                   triggering a ~30-60 minute rebuild via Section 7b.
#
# Default both False. FORCE_REPREPROCESS kept as a legacy alias that turns
# ON the Drive wipe (cell 12 honors it for backward compatibility) so older
# muscle memory still works, but new sessions should prefer the explicit
# WIPE_* flags.
# WIPE_LOCAL_CACHE   = False
# WIPE_DRIVE_CACHE   = False
# FORCE_REPREPROCESS = False  # legacy alias: True implies WIPE_DRIVE_CACHE=True
# if FORCE_REPREPROCESS:
    # WIPE_DRIVE_CACHE = True
# print(f'WIPE_LOCAL_CACHE={WIPE_LOCAL_CACHE}, WIPE_DRIVE_CACHE={WIPE_DRIVE_CACHE} '
      # f'(actual wipe happens in cell 12 after Drive mount + config)')


import pathlib
for sub in ['', 'checkpoints', 'results', 'gifs', 'feasibility']:
    pathlib.Path(f'{DRIVE_OUT_MOTION}/{sub}').mkdir(parents=True, exist_ok=True)
print(f'\nMotion-rep output dir: {DRIVE_OUT_MOTION}')

## 2. Setup

Mount Google Drive, clone the team repo (PAT from Colab Secrets), build a Python 3.9 venv, and install dependencies. The fast path reuses a captured `working_requirements.txt` from Drive; a sanity check runs before training.

In [ ]:
# Force-fresh clone with PAT  clears the broken repo and pulls main fresh.
import subprocess, shutil, pathlib, os
from google.colab import userdata

REPO_PATH = '/content/CS269FlowPlannerProject'
PAT = userdata.get('token')  # Colab secret named 'token' (same one cell 12 uses)
assert PAT, "Colab secret 'token' is missing. Set it in the left sidebar 'Secrets' panel."

if pathlib.Path(REPO_PATH).exists():
    shutil.rmtree(REPO_PATH)

# Build the authenticated URL but never print it. Use git credential.helper
# so the PAT isn't on argv and isn't written to the repo's .git/config.
url_with_pat = f'https://{PAT}@github.com/wimaan3/CS269FlowPlannerProject.git'
try:
    subprocess.run(
        ['git', 'clone', '--depth', '1', '-b', 'main', url_with_pat, REPO_PATH],
        check=True, capture_output=True, text=True,
    )
finally:
    # Scrub the URL from local memory ASAP
    del url_with_pat

# Reset origin to the no-PAT URL so the PAT doesn't end up in .git/config
subprocess.run(
    ['git', '-C', REPO_PATH, 'remote', 'set-url', 'origin',
     'https://github.com/wimaan3/CS269FlowPlannerProject.git'],
    check=True,
)
subprocess.run(['git', '-C', REPO_PATH, 'log', '-1', '--oneline'], check=True)
print('repo re-cloned fresh on main')


In [ ]:
import os, pathlib, subprocess

IS_COLAB_ENTERPRISE = os.environ.get('VERTEX_PRODUCT') == 'COLAB_ENTERPRISE'

if IS_COLAB_ENTERPRISE:
    # Drive mount is disabled in Colab Enterprise. Set up a local mimic of
    # /content/drive/MyDrive/cs269 so the rest of the notebook works
    # unchanged. Cache is reused if the runtime already has it; otherwise
    # zips are downloaded from the public S3 bucket and preprocess runs.
    print("Colab Enterprise detected - using local Drive mimic + S3 fallback")
    DRIVE_FAKE = "/content/drive/MyDrive/cs269"
    for sub in ["nuplan_zips", "checkpoints", "results", "experiments"]:
        pathlib.Path(f"{DRIVE_FAKE}/{sub}").mkdir(parents=True, exist_ok=True)

    # Download zips if missing
    S3_BASE = "s3://motional-nuplan/public/nuplan-v1.1"
    zips = [
        (f"{S3_BASE}/nuplan-v1.1_mini.zip", f"{DRIVE_FAKE}/nuplan_zips/nuplan-v1.1_mini.zip", 8_550_100_030),
        (f"{S3_BASE}/nuplan-maps-v1.0.zip", f"{DRIVE_FAKE}/nuplan_zips/nuplan-maps-v1.0.zip", 971_557_640),
    ]
    need_aws = any(not os.path.exists(d) or os.path.getsize(d) < e * 0.99 for _, d, e in zips)
    if need_aws:
        subprocess.run(["pip", "install", "awscli", "-q"], check=True)
        for src, dst, expected in zips:
            if os.path.exists(dst) and os.path.getsize(dst) >= expected * 0.99:
                continue
            print(f"  downloading {os.path.basename(dst)}")
            subprocess.run(["aws", "s3", "cp", "--no-sign-request", src, dst], check=True)
    print("  zips ready")

    # Set the flag downstream cells expect
    _CONFIG_CELL_RAN = True
else:
    # Regular Colab Pro: use the standard drive.mount
    from google.colab import drive
    drive.mount("/content/drive")
    _CONFIG_CELL_RAN = True
    print("Drive mounted at /content/drive")


In [ ]:
import pathlib

# Auto-discover the actual cache folder names on this Drive. If a populated
# cache is found, use it. If not, fall back to canonical names and let the
# bootstrap below extract+preprocess from raw zips in nuplan_zips/.

_DRIVE = pathlib.Path(DRIVE_FOLDER)

# Preprocessed cache
_pre_candidates = ['preprocessed_cache_1500', 'preprocessed_cache_full', 'preprocessed_cache']
_DRIVE_PREPROCESSED = None
for c in _pre_candidates:
    p = _DRIVE / c
    if p.exists() and any(True for _ in p.iterdir()):
        _DRIVE_PREPROCESSED = str(p)
        print(f'Found populated cache:  {_DRIVE_PREPROCESSED}')
        break
if _DRIVE_PREPROCESSED is None:
    # No populated cache — bootstrap will rebuild from zips below.
    # Pick a canonical name as the rebuild target.
    _DRIVE_PREPROCESSED = str(_DRIVE / 'preprocessed_cache')
    print(f'No populated cache yet — bootstrap will extract+preprocess to: {_DRIVE_PREPROCESSED}')
    # Verify zips are present so we do not proceed pointlessly.
    _zips_dir = _DRIVE / 'nuplan_zips'
    _zips = sorted(_zips_dir.glob('*.zip')) if _zips_dir.exists() else []
    if len(_zips) < 2:
        raise RuntimeError(
            f'No cache and insufficient zips at {_zips_dir} ({len(_zips)} found). '
            f'Need nuplan-maps-v1.0.zip and nuplan-v1.1_mini.zip.'
        )
    print(f'  zips ready: {[z.name for z in _zips]}')

# Heldout cache
_held_candidates = ['heldout_cache', 'heldout_cache_300', 'heldout_cache_500']
_DRIVE_HELDOUT = None
for c in _held_candidates:
    p = _DRIVE / c
    if p.exists() and any(True for _ in p.iterdir()):
        _DRIVE_HELDOUT = str(p)
        print(f'Found heldout cache:    {_DRIVE_HELDOUT}')
        break
if _DRIVE_HELDOUT is None:
    _DRIVE_HELDOUT = str(_DRIVE / 'heldout_cache_300')
    print(f'No heldout cache yet — bootstrap will build to: {_DRIVE_HELDOUT}')

# Overwrite the variables the rest of the notebook uses.
DRIVE_PREPROCESSED = _DRIVE_PREPROCESSED
DRIVE_HELDOUT      = _DRIVE_HELDOUT
try:
    DRIVE_PREPROCESSED_1500 = DRIVE_PREPROCESSED
except Exception:
    pass

print(f'\nDRIVE_PREPROCESSED = {DRIVE_PREPROCESSED}')
print(f'DRIVE_HELDOUT      = {DRIVE_HELDOUT}')

In [ ]:
import pathlib, subprocess, shutil

def _sh(cmd, **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if r.returncode != 0:
        print('STDOUT:', r.stdout); print('STDERR:', r.stderr)
        raise RuntimeError(f'Command failed: {cmd[:4]}...')
    return r

# Validate existing venv before reusing it (catches corrupted half-installs)
venv_ok = False
if pathlib.Path(VENV).exists():
    probe = subprocess.run([PYTHON, '--version'], capture_output=True, text=True)
    venv_ok = (probe.returncode == 0 and '3.9' in probe.stdout)
    if not venv_ok:
        print(f'Existing venv at {VENV} is broken or wrong version — rebuilding')
        shutil.rmtree(VENV, ignore_errors=True)

if not venv_ok:
    print('Installing python3.9 (deadsnakes PPA + apt)...')
    # python3.9 is NOT in Ubuntu 22.04 default sources — need deadsnakes PPA.
    _sh(['sudo', 'add-apt-repository', '-y', 'ppa:deadsnakes/ppa'])
    _sh(['sudo', 'apt-get', 'update', '-qq'])
    _sh(['sudo', 'apt-get', 'install', '-y', 'python3.9', 'python3.9-venv', 'python3.9-dev'])
    # Sanity: python3.9 binary must actually exist now
    py39_probe = subprocess.run(['python3.9', '--version'], capture_output=True, text=True)
    assert py39_probe.returncode == 0, f'python3.9 still not installed after apt: {py39_probe.stderr}'
    print(f'  apt OK: {py39_probe.stdout.strip()}')
    _sh(['python3.9', '-m', 'venv', VENV])
    _sh([PYTHON, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel', '-q'])
else:
    print(f'venv exists and is valid at {VENV}')

ver = subprocess.run([PYTHON, '--version'], capture_output=True, text=True).stdout.strip()
assert '3.9' in ver, f'Wrong Python version in venv: {ver} (expected 3.9.x)'
print(ver)


In [ ]:
# Clone nuplan-devkit (required for the editable install in the next cell)
import pathlib, subprocess
if not pathlib.Path(NUPLAN_DEVKIT_DIR).exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/motional/nuplan-devkit.git',
         NUPLAN_DEVKIT_DIR],
        check=True,
    )
    print(f'cloned nuplan-devkit -> {NUPLAN_DEVKIT_DIR}')
else:
    print(f'nuplan-devkit already present at {NUPLAN_DEVKIT_DIR}')

In [ ]:
import pathlib, time

# FUSE-lag retry: DRIVE_REQUIREMENTS may briefly read as missing right after
# mount even when present on Drive.
for _ in range(5):
    if pathlib.Path(DRIVE_REQUIREMENTS).exists():
        break
    time.sleep(1)

FAST = pathlib.Path(DRIVE_REQUIREMENTS).exists()
if FAST:
    # Probe size — empty file from a failed prior capture should NOT trigger FAST
    size = pathlib.Path(DRIVE_REQUIREMENTS).stat().st_size
    if size < 100:
        print(f'  {DRIVE_REQUIREMENTS} is suspiciously small ({size} bytes) — forcing SLOW path')
        FAST = False

print(f'Install path: {"FAST" if FAST else "SLOW"}')
if FAST:
    print(f'  Using cached requirements from {DRIVE_REQUIREMENTS}')
else:
    print('  Will install from scratch (~15-20 min)')

# Constraints file pinning numpy<2 (must persist across all pip invocations).
# Live under LOCAL_ROOT, not /tmp (Colab wipes /tmp between reconnects).
pathlib.Path(LOCAL_ROOT).mkdir(parents=True, exist_ok=True)
with open(CONSTRAINTS, 'w') as f:
    f.write('# pip constraints — pins numpy<2 for nuplan/flow_planner compatibility\nnumpy<2\n')

# FAST path: install from captured working_requirements.txt with --no-deps,
# then run `pip check` to catch missing transitive deps loudly.
# SLOW path: install nuplan-devkit requirements with filtering for broken RC pins.
import pathlib, subprocess

def _pip(args, **kw):
    r = subprocess.run([PIP] + list(args), capture_output=True, text=True, **kw)
    if r.returncode != 0:
        print('STDOUT (tail):', '\n'.join(r.stdout.splitlines()[-25:]))
        print('STDERR (tail):', '\n'.join(r.stderr.splitlines()[-25:]))
        raise RuntimeError(f'pip failed: {args[:2]}')
    # On success print just the last few lines for visibility
    tail = '\n'.join(r.stdout.splitlines()[-10:])
    print(tail)
    return r

if FAST:
    # Strip editable lines (we redo those separately) and filter our own packages
    clean = pathlib.Path('/tmp/working_clean_reqs.txt')
    with open(DRIVE_REQUIREMENTS) as fin, clean.open('w') as fout:
        for line in fin:
            s = line.strip()
            if (not s or s.startswith('#') or s.startswith('-e ') or '@ file://' in s
                or 'flow_planner' in s or 'nuplan-devkit' in s or 'diffusion_planner' in s):
                continue
            fout.write(line)
    assert clean.stat().st_size > 100, f'Cleaned reqs file is empty/tiny: {clean}'
    _pip(['install', '-r', str(clean), '--no-deps'])
    # CRITICAL: --no-deps means we must verify the dep graph is satisfied
    chk = subprocess.run([PIP, 'check'], capture_output=True, text=True)
    if chk.returncode != 0:
        print('pip check found missing/broken deps (FAST path is incomplete):')
        print(chk.stdout)
        # Don't abort — but try to repair by installing missing deps with constraints
        print('Attempting to repair by installing flow_planner requirements with constraints...')
        _pip(['install', '-r', f'{FP_DIR}/requirements.txt', '-c', CONSTRAINTS])
        chk2 = subprocess.run([PIP, 'check'], capture_output=True, text=True)
        if chk2.returncode != 0:
            print('pip check STILL failing after repair:')
            print(chk2.stdout)
            print('You can proceed (sanity check in cell 21 will catch flow_planner import failures)')
            print('or delete DRIVE_REQUIREMENTS and re-run to get a fresh SLOW install.')
        else:
            print('Repair OK — dep graph now satisfied.')
    else:
        print('pip check OK — all deps satisfied.')
else:
    # SLOW path: filter nuplan-devkit requirements (drop broken hydra/omegaconf RC pins)
    req_in = f'{NUPLAN_DEVKIT_DIR}/requirements.txt'
    req_out = '/tmp/filtered_requirements.txt'
    bad_pins = ['hydra-core', 'omegaconf']
    with open(req_in) as f, open(req_out, 'w') as g:
        for line in f:
            # Strip leading whitespace before the prefix check so '  hydra-core==X' is caught
            stripped = line.lstrip().lower()
            if stripped.startswith('#'):
                g.write(line); continue
            if not any(stripped.startswith(b) for b in bad_pins):
                g.write(line)
    _pip(['install', '-r', req_out, '-c', CONSTRAINTS])
    _pip(['install', 'hydra-core>=1.2,<1.4', 'omegaconf>=2.2,<2.4', '-c', CONSTRAINTS])
    _pip(['install', '-r', f'{FP_DIR}/requirements.txt', '-c', CONSTRAINTS])

# Editable install of nuplan-devkit + flow_planner so our modifications take effect
!{PIP} install -e {NUPLAN_DEVKIT_DIR} --no-deps 2>&1 | tail -3
!{PIP} install -e {FP_DIR} --no-deps 2>&1 | tail -3

# Defense-in-depth: pin numpy<2
import subprocess
r = subprocess.run([PIP, 'install', 'numpy<2', '--force-reinstall', '--no-deps', '-q'],
                   capture_output=True, text=True)
if r.returncode != 0:
    print('STDOUT:', r.stdout); print('STDERR:', r.stderr)
    raise RuntimeError('numpy<2 pin failed')
ver = subprocess.run([PYTHON, '-c', 'import numpy; print(f"numpy {numpy.__version__}")'],
                     capture_output=True, text=True).stdout.strip()
print(ver)
assert ver.startswith('numpy 1.'), f'Wrong numpy version: {ver} (expected 1.x)'


In [ ]:
# Implementation lives at scripts/check_sanity.py — runs in the venv's
# Python 3.9 because the notebook kernel (Colab 3.11) cannot import
# nuplan / flow_planner.
import os, subprocess

r = subprocess.run(
    [PYTHON, f'{REPO_DIR}/scripts/check_sanity.py'],
    capture_output=True, text=True,
    env={**os.environ, 'FP_DIR': FP_DIR},
)
print(r.stdout)
all_ok = r.returncode == 0
if not all_ok:
    print('STDERR:'); print(r.stderr)
print(f'\nall_ok = {all_ok}')
assert all_ok, 'Sanity check failed — fix above before continuing'


In [ ]:
if all_ok and not FAST:
    print(f'Capturing {DRIVE_REQUIREMENTS}...')
    !{PIP} freeze > {DRIVE_REQUIREMENTS}
    print('Done')
else:
    print('Skipping (FAST path already used existing file)')


## 3. Data

Load the preprocessed `.npz` cache from Drive if present; otherwise unzip raw nuPlan and run `preprocess`. Then validate the cache and write `train_manifest.json`. A held-out cache is built from the val-log split when raw data is available — eval on held-out is skipped cleanly if not.

In [ ]:
import pathlib, time, subprocess, shutil

# Honour the scale-aware Drive cache picked in cell 6 (DRIVE_PREPROCESSED).
# Older sessions still wrote to preprocessed_cache_1500; keep the legacy var
# as a synonym for any older cell references downstream.
drive_cache = pathlib.Path(DRIVE_PREPROCESSED)
local_cache = pathlib.Path(LOCAL_CACHE)
local_cache.mkdir(parents=True, exist_ok=True)

# train_cache_ready signals to Section 7b whether the fallback unzip +
# preprocess should run. True after a successful Drive-cache load (even if
# methodology-warning fired); False if the Drive cache is missing.
train_cache_ready = False

if drive_cache.exists():
    drive_npz = sorted(drive_cache.glob('*.npz'))
    print(f'Drive cache: {len(drive_npz)} .npz files at {drive_cache}')

    if len(drive_npz) == 0:
        raise RuntimeError(f'Drive cache {drive_cache} exists but contains no .npz files')

    # Scale-up fix: replace the previous per-file `!cp` loop (~100-300ms/file
    # via FUSE + bash subprocess fork, i.e. tens of minutes at ~10k files)
    # with a single rsync call. rsync's --ignore-existing handles "what to
    # copy" in one pass, so we also skip the upfront 9999-call stat() scan.
    t = time.time()
    print(f'Copying via rsync (Drive -> local) ...')
    r = subprocess.run(
        ['rsync', '-a', '--ignore-existing', '--info=stats2',
         f'{drive_cache}/', f'{local_cache}/'],
        capture_output=True, text=True,
    )
    # rsync exit 0 = ok, 24 = "some files vanished" (benign for Drive FUSE).
    if r.returncode not in (0, 24):
        print(r.stdout); print(r.stderr)
        raise RuntimeError(f'rsync failed with exit code {r.returncode}')
    print(r.stdout.strip().splitlines()[-3:] if r.stdout else '')
    print(f'rsync done in {time.time()-t:.1f}s')

    # Also copy the preprocess_manifest.json (written next to the .npz files
    # by flow_planner/run_script/preprocess.py). The methodology-compliance
    # check below reads it from local_cache; without this copy the check
    # always fires the WARNING branch even when the cache was actually
    # methodology-compliant.
    drive_manifest = drive_cache / 'preprocess_manifest.json'
    if drive_manifest.exists():
        shutil.copy(str(drive_manifest), str(local_cache / 'preprocess_manifest.json'))
        print('Copied preprocess_manifest.json from Drive')

    # Post-copy integrity: catches truncated / zero-byte files. We compare
    # against Drive's per-file sizes only when the index is cheap; at full
    # scale we settle for zero-byte detection because the full stat scan is
    # itself expensive on Drive FUSE.
    local_npz_post = sorted(local_cache.glob('*.npz'))
    zero_byte = [p for p in local_npz_post if p.stat().st_size == 0]
    if zero_byte:
        raise RuntimeError(
            f'Post-copy integrity check failed: {len(zero_byte)} zero-byte files. '
            f'Delete those files locally and re-run this cell.'
        )
    print(f'Post-copy OK: {len(local_npz_post)} files in {local_cache}')

    # --- Methodology compliance check ---
    # The Drive cache may have been generated before the Phase 2 log-disjoint
    # split was introduced. If the manifest says it was NOT preprocessed with
    # --log_names_key=train pointing to LOG_SPLIT_JSON, we warn loudly (but
    # do not delete or auto-regenerate - the user decides).
    import json as _json
    manifest_path = local_cache / 'preprocess_manifest.json'
    methodology_ok = False
    if manifest_path.exists():
        _m = _json.loads(manifest_path.read_text())
        _lnj = _m.get('log_names_json')
        _lnk = _m.get('log_names_key')
        if _lnj and _lnk == 'train':
            methodology_ok = True
            print(f'Methodology: cache built with log_names_json={_lnj} key={_lnk!r} OK')

    if not methodology_ok:
        print()
        print('!!!!!! METHODOLOGY WARNING !!!!!!')
        if manifest_path.exists():
            print(f'!!! preprocess_manifest.json present but log_names_key != "train"')
            print(f'!!! manifest log_names_json: {_m.get("log_names_json")!r}')
            print(f'!!! manifest log_names_key:  {_m.get("log_names_key")!r}')
        else:
            print('!!! No preprocess_manifest.json in cache - likely pre-methodology cache.')
        print('!!! This cache may contain scenarios from val logs (DATA LEAKAGE).')
        print('!!! For methodology-compliant training:')
        print(f'!!!   1. Delete the Drive cache {DRIVE_PREPROCESSED}')
        print('!!!   2. Re-run cell 22 (will fall through to Section 7b which uses')
        print('!!!      --log_names_key=train and produces a clean train cache)')
        print('!!! Or proceed with the existing cache and disclose the')
        print('!!! limitation in the report.')
        print()

    # Cache was loaded from Drive (regardless of methodology status - the user
    # has been warned and can decide). Section 7b will skip.
    train_cache_ready = True
else:
    print(f'Drive cache {drive_cache} NOT found - will fall back to unzip + preprocess')
    print('(Section 7b - cells 27 and 28 - will run; Section 7c will validate)')

# Fallback path: unzip nuPlan from Drive. Runs only if cell 22 could not load
# the Drive-cache shortcut. Uses an explicit filename → target mapping rather
# than substring matching so a future Drive zip named "mini-with-maps.zip" or
# similar can\'t silently route to the wrong directory.
if not train_cache_ready:
    import time, pathlib, zipfile

    ZIP_TARGETS = {
        'nuplan-maps-v1.0.zip': LOCAL_MAPS,
        'nuplan-v1.1_mini.zip': LOCAL_LOGS,
    }

    pathlib.Path(LOCAL_MAPS).mkdir(parents=True, exist_ok=True)
    pathlib.Path(LOCAL_LOGS).mkdir(parents=True, exist_ok=True)
    local_zip_dir = pathlib.Path(f'{LOCAL_ROOT}/_zips')
    local_zip_dir.mkdir(parents=True, exist_ok=True)

    for z in sorted(pathlib.Path(DRIVE_ZIPS).glob('*.zip')):
        if z.name not in ZIP_TARGETS:
            print(f'  SKIP {z.name}: not in ZIP_TARGETS allowlist')
            continue
        target = ZIP_TARGETS[z.name]
        # If target is already populated (previous extraction completed),
        # skip the unzip step — saves several minutes on re-runs.
        if any(pathlib.Path(target).iterdir()):
            print(f'  {z.name}: {target} already populated, skipping extract')
            continue
        dest = local_zip_dir / z.name
        if not dest.exists():
            t = time.time()
            # rsync is more reliable than shutil.copy for large Drive files
            !rsync --inplace {z} {dest}
            print(f'Copied {z.name} in {time.time()-t:.1f}s')
        try:
            with zipfile.ZipFile(dest) as zf:
                print(f'Unzipping {z.name} -> {target} ...')
                t = time.time()
                zf.extractall(target)
                print(f'  done in {time.time()-t:.1f}s')
        except zipfile.BadZipFile:
            print(f'  ERROR: {dest} is corrupted. Try re-uploading to Drive.')
else:
    print('Drive cache loaded successfully in cell 22 — fallback unzip not needed.')


In [ ]:
# AUTO-FLATTEN: nuPlan zips often extract into nested dirs (e.g.
# LOCAL_MAPS/maps/... instead of LOCAL_MAPS/...). The preprocess step in
# the next cells needs the CANONICAL paths, so we detect the nested
# structure once and move it up. Runs unconditionally — if the structure
# is already flat or the Drive-cache fast-path was used, it's a no-op.
import pathlib, shutil

def _flatten_if_nested(parent: pathlib.Path, expected_marker: str):
    """If parent is empty / lacks expected_marker but parent/<subdir>/expected_marker
    exists, move parent/<subdir>/* up into parent."""
    if not parent.exists():
        return False
    if (parent / expected_marker).exists():
        return False  # already flat
    # Look one level deep for the marker
    for sub in parent.iterdir():
        if sub.is_dir() and (sub / expected_marker).exists():
            print(f'  flattening {sub} -> {parent}')
            for item in list(sub.iterdir()):
                target = parent / item.name
                if target.exists():
                    continue  # don't clobber
                shutil.move(str(item), str(target))
            try:
                sub.rmdir()
            except OSError:
                pass
            return True
    return False

# Maps: expect LOCAL_MAPS/nuplan-maps-v1.0.json directly
maps_p = pathlib.Path(LOCAL_MAPS)
if _flatten_if_nested(maps_p, 'nuplan-maps-v1.0.json'):
    print(f'  flattened maps; now have: {sorted(p.name for p in maps_p.iterdir())[:5]}')

# Logs: expect LOCAL_LOGS/*.db directly. Walk two levels deep because zips
# sometimes nest as logs/data/cache/mini/*.db.
logs_p = pathlib.Path(LOCAL_LOGS)
if logs_p.exists() and not any(logs_p.glob('*.db')):
    candidates = list(logs_p.rglob('*.db'))
    if candidates:
        first_dir = candidates[0].parent
        print(f'  flattening logs from {first_dir} -> {logs_p}')
        for db in candidates:
            target = logs_p / db.name
            if not target.exists():
                shutil.move(str(db), str(target))
        # Best-effort cleanup of empty intermediate dirs
        for sub in sorted([p for p in logs_p.rglob('*') if p.is_dir()], key=lambda p: -len(p.parts)):
            try: sub.rmdir()
            except OSError: pass

print(f'After flatten:')
print(f'  {LOCAL_MAPS}/nuplan-maps-v1.0.json exists: {(maps_p/"nuplan-maps-v1.0.json").exists()}')
print(f'  {LOCAL_LOGS}/*.db count: {len(list(logs_p.glob("*.db")))}')

# the chronic 'preprocess produced 0 .npz files, then validate fails
# with cryptic message' failure mode.
import pathlib as _p
if not train_cache_ready:
    _maps = _p.Path(f'{LOCAL_MAPS}/nuplan-maps-v1.0.json')
    _dbs  = list(_p.Path(LOCAL_LOGS).glob('*.db'))
    assert _maps.exists(), (
        f'Cannot run preprocess: maps file missing at {_maps}. '
        f'Either the extract+flatten cells did not run, or '
        f'nuplan-maps-v1.0.zip was missing from DRIVE_ZIPS={DRIVE_ZIPS}.'
    )
    assert len(_dbs) > 0, (
        f'Cannot run preprocess: 0 .db files at {LOCAL_LOGS}. '
        f'Either the extract+flatten cells did not run, or '
        f'nuplan-v1.1_mini.zip was missing from DRIVE_ZIPS={DRIVE_ZIPS}.'
    )
    print(f'Pre-preprocess OK: maps present, {len(_dbs)} .db files at {LOCAL_LOGS}')

# Fallback preprocess: train logs only via LOG_SPLIT_JSON's "train" key
# (54 of the 64 mini logs, deterministic given seed=42). Runs only if
# cell 22 could not load the Drive-cache shortcut. The training manifest
# is written by Section 7c, NOT here - this cell only produces the .npz
# files. Previous version wrote its own manifest, which clobbered the
# quarantined one Section 7c produces.
if not train_cache_ready:
    import os, pathlib
    os.environ['NUPLAN_DATA_ROOT'] = LOCAL_NUPLAN
    os.environ['NUPLAN_MAPS_ROOT'] = LOCAL_MAPS
    os.environ['NUPLAN_EXP_ROOT']  = LOCAL_EXP

    # Ensure the log-disjoint split JSON exists. Deterministic given LOCAL_LOGS
    # contents + seed=42 - re-runs produce the same file byte-for-byte. If a
    # committed docs/log_split_mini_seed42.json already exists, this is a no-op.
    split_path = pathlib.Path(LOG_SPLIT_JSON)
    if not split_path.exists():
        print(f'Generating {split_path} from {LOCAL_LOGS}...')
        !{PYTHON} {REPO_DIR}/scripts/generate_log_split.py \
            --logs_dir {LOCAL_LOGS} \
            --output {split_path}

    # CONTRACT: only pass --total_scenarios when truthy (non-None, > 0).
    # The preprocess.py CLI default is None=unlimited; passing 0 or '' to
    # argparse converts to int(0) which some nuPlan versions interpret as
    # zero-scenarios (NOT unlimited). The cell 6 assertion already
    # rejects TOTAL_SCENARIOS=0, but explicitly omit the flag for clarity.
    _cap_flag = f'--total_scenarios {TOTAL_SCENARIOS}' if TOTAL_SCENARIOS else ''

    %cd {FP_DIR}
    !{PYTHON} -m flow_planner.run_script.preprocess \
        --data_path {LOCAL_LOGS} \
        --map_path {LOCAL_MAPS} \
        --save_path {LOCAL_CACHE} \
        {_cap_flag} \
        --log_names_json {LOG_SPLIT_JSON} \
        --log_names_key train \
        --seed {RUN_SEED}

    import pathlib
    npz_files = sorted(pathlib.Path(LOCAL_CACHE).glob('*.npz'))
    print(f'{len(npz_files)} .npz files in {LOCAL_CACHE} (manifest written by Section 7c)')

    # Post-preprocess count check (silent-truncation guard for the fallback
    # path; cell 27 only guards the Drive-cache path).
    if TOTAL_SCENARIOS is None:
        _floor = 5000
        assert len(npz_files) >= _floor, (
            f'Fallback preprocess produced only {len(npz_files)} files; '
            f'expected >= {_floor} for the full mini-split. '
            f'Check preprocess_failures.json in {LOCAL_CACHE} for per-scenario errors.'
        )
    else:
        if len(npz_files) == TOTAL_SCENARIOS:
            print(f'!!! NOTE: file count exactly equals TOTAL_SCENARIOS={TOTAL_SCENARIOS}. '
                  f'The safety cap was HIT - you may be silently truncating. '
                  f'Set TOTAL_SCENARIOS=None in cell 6 for the full mini-split.')
else:
    print('Drive cache loaded successfully in cell 22 - fallback preprocess not needed.')


In [ ]:
# diagnostic of where files actually are so the user can fix the path
# rather than guess what went wrong.
import pathlib as _p
_local = _p.Path(LOCAL_CACHE)
_npz_count = len(list(_local.glob('*.npz'))) if _local.exists() else 0
if _npz_count == 0:
    _drive_c = _p.Path(DRIVE_PREPROCESSED_1500)
    _drive_z = _p.Path(DRIVE_ZIPS)
    print('='*60)
    print('CACHE DIAGNOSTIC (training cache is empty)')
    print('='*60)
    print(f'LOCAL_CACHE             = {LOCAL_CACHE}')
    print(f'  exists                = {_local.exists()}')
    print(f'  .npz count            = {_npz_count}')
    print(f'DRIVE_PREPROCESSED_1500 = {DRIVE_PREPROCESSED_1500}')
    print(f'  exists                = {_drive_c.exists()}')
    if _drive_c.exists():
        print(f'  .npz count            = {len(list(_drive_c.glob("*.npz")))}')
    print(f'DRIVE_ZIPS              = {DRIVE_ZIPS}')
    print(f'  exists                = {_drive_z.exists()}')
    if _drive_z.exists():
        zips = sorted(_drive_z.glob('*.zip'))
        print(f'  .zip files            = {[z.name for z in zips]}')
    print(f'LOCAL_LOGS              = {LOCAL_LOGS}')
    _logs_p = _p.Path(LOCAL_LOGS)
    if _logs_p.exists():
        print(f'  .db count             = {len(list(_logs_p.glob("*.db")))}')
        nested_dbs = list(_logs_p.rglob('*.db'))
        if nested_dbs and not any(_logs_p.glob('*.db')):
            print(f'  NESTED .db files found at: {nested_dbs[0].parent}')
            print(f'  -> auto-flatten cell did not run or did not catch this layout')
    print(f'LOCAL_MAPS              = {LOCAL_MAPS}')
    _maps_p = _p.Path(LOCAL_MAPS)
    if _maps_p.exists():
        print(f'  has nuplan-maps-v1.0.json: {(_maps_p/"nuplan-maps-v1.0.json").exists()}')
        nested = list(_maps_p.rglob('nuplan-maps-v1.0.json'))
        if nested and not (_maps_p/"nuplan-maps-v1.0.json").exists():
            print(f'  NESTED maps json found at: {nested[0]}')
    print('='*60)
    print('Likely fixes:')
    print('  - If Drive zips are missing: upload nuplan-maps-v1.0.zip and')
    print('    nuplan-v1.1_mini.zip to', DRIVE_ZIPS)
    print('  - If files are at nested paths: re-run the auto-flatten cell')
    print('    that follows the extract cell, then re-run preprocess.')
    print('  - If you have a v6 Drive cache at a different path: update')
    print('    DRIVE_PREPROCESSED_1500 in the config cell.')
    print('='*60)
    raise RuntimeError(f'Training cache validation failed - 0 .npz files at {LOCAL_CACHE}. See diagnostic above.')

# Full per-file validation of the preprocessed cache. Implementation in
# scripts/validate_npz_cache.py — same script Section 8c uses for the
# held-out cache, so the validation rules can never drift between the
# two paths. Quarantines bad files from the training manifest; raises
# if more than 5% of files fail (systemic problem, not per-file accident).
import os, subprocess
r = subprocess.run(
    [PYTHON, f'{REPO_DIR}/scripts/validate_npz_cache.py'],
    capture_output=True, text=True,
    env={**os.environ,
         'CACHE_DIR':       LOCAL_CACHE,
         'OUTPUT_MANIFEST': f'{LOCAL_CACHE}/diffusion_planner_training.json',
         'FAIL_THRESHOLD':  '0.05'},
)
print(r.stdout)
if r.returncode != 0:
    print('STDERR:'); print(r.stderr)
    raise RuntimeError('Training cache validation failed — stop before training')


In [ ]:
# Set up raw nuPlan data + ensure the log-disjoint split JSON exists.
# This cell runs ALWAYS — Section 7's main path (Drive cache) bypasses
# unzipping for training, but the held-out path needs raw data either way.
import pathlib, zipfile

HELDOUT_AVAILABLE = False  # True only if BOTH raw data + log split are ready

# Explicit filename -> target mapping. Drop the substring-match approach
# (a future zip named "nuplan-with-maps-bundled.zip" would silently route
# to the wrong directory under substring routing).
ZIP_TARGETS = {
    'nuplan-maps-v1.0.zip': LOCAL_MAPS,
    'nuplan-v1.1_mini.zip': LOCAL_LOGS,
}

logs_dir = pathlib.Path(LOCAL_LOGS)
maps_dir = pathlib.Path(LOCAL_MAPS)
have_raw = logs_dir.exists() and any(logs_dir.iterdir()) and maps_dir.exists() and any(maps_dir.iterdir())

if have_raw:
    print('Raw nuPlan data already present locally — proceeding')
else:
    drive_zips = sorted(pathlib.Path(DRIVE_ZIPS).glob('*.zip'))
    print(f'Attempting to unzip {len(drive_zips)} zip(s) from Drive...')
    local_zip_dir = pathlib.Path(f'{LOCAL_ROOT}/_zips')
    local_zip_dir.mkdir(parents=True, exist_ok=True)
    pathlib.Path(LOCAL_MAPS).mkdir(parents=True, exist_ok=True)
    pathlib.Path(LOCAL_LOGS).mkdir(parents=True, exist_ok=True)

    extracted_any = False
    for z in drive_zips:
        if z.name not in ZIP_TARGETS:
            print(f'  SKIP {z.name}: not in ZIP_TARGETS allowlist')
            continue
        target = ZIP_TARGETS[z.name]
        # If target is already populated (previous extraction completed),
        # skip the unzip step — saves several minutes on re-runs.
        if any(pathlib.Path(target).iterdir()):
            print(f'  {z.name}: {target} already populated, skipping extract')
            extracted_any = True
            continue
        dest = local_zip_dir / z.name
        if not dest.exists():
            !rsync --inplace {z} {dest}
        try:
            with zipfile.ZipFile(dest) as zf:
                print(f'  Unzipping {z.name} -> {target} ...')
                zf.extractall(target)
                extracted_any = True
        except zipfile.BadZipFile:
            # Continue with whatever other zips work rather than aborting
            # the whole held-out path on one bad zip.
            print(f'  CORRUPTED: {z.name} — skipping; held-out may be incomplete')

    have_raw = extracted_any and any(logs_dir.iterdir()) and any(maps_dir.iterdir())

if have_raw:
    # Generate the log split JSON if it isn't already present. Deterministic
    # given LOCAL_LOGS + seed=42 — re-runs produce the same file. If a
    # committed docs/log_split_mini_seed42.json already exists, this is a no-op.
    split_path = pathlib.Path(LOG_SPLIT_JSON)
    if not split_path.exists():
        print(f'\nGenerating log split {split_path} from {LOCAL_LOGS}...')
        !{PYTHON} {REPO_DIR}/scripts/generate_log_split.py \
            --logs_dir {LOCAL_LOGS} \
            --output {split_path}
    else:
        print(f'\nLog split already present: {split_path}')
    HELDOUT_AVAILABLE = True
    print('Raw data + log split ready — proceeding to held-out preprocess')
else:
    print('\nHeld-out skipped. Will use training-set ADE/FDE only in final table.')

# Preprocess HELD-OUT scenarios from the val logs only. The held-out cache
# draws from LOG_SPLIT_JSON's "val" key (10 of the 64 mini logs, disjoint
# from the "train" key cell 28 of Section 7b uses). This is the Phase 2
# methodology requirement - no log overlap with the training cache.
#
# The training manifest is written by cell 33 (Section 8c) AFTER per-file
# validation, NOT here. The previous version of this cell wrote its own
# all-files manifest, which would let bad files into the held-out eval.
import os, pathlib

if HELDOUT_AVAILABLE:
    existing = sorted(pathlib.Path(LOCAL_HELDOUT_CACHE).glob('*.npz'))
    expected = HELDOUT_SCENARIOS if (HELDOUT_SCENARIOS and HELDOUT_SCENARIOS > 0) else len(existing)

    if existing and len(existing) >= int(0.95 * expected):
        print(f'Held-out cache already populated ({len(existing)} files), skipping preprocess')
        print('(Section 8c will re-validate and write the manifest)')
    else:
        os.environ['NUPLAN_DATA_ROOT'] = LOCAL_NUPLAN
        os.environ['NUPLAN_MAPS_ROOT'] = LOCAL_MAPS
        os.environ['NUPLAN_EXP_ROOT']  = LOCAL_EXP

        # Mirror cell 32's CONTRACT: only pass --total_scenarios when truthy.
        _heldout_cap_flag = f'--total_scenarios {HELDOUT_SCENARIOS}' if HELDOUT_SCENARIOS else ''

        %cd {FP_DIR}
        !{PYTHON} -m flow_planner.run_script.preprocess \
            --data_path {LOCAL_LOGS} \
            --map_path {LOCAL_MAPS} \
            --save_path {LOCAL_HELDOUT_CACHE} \
            {_heldout_cap_flag} \
            --log_names_json {LOG_SPLIT_JSON} \
            --log_names_key val \
            --seed {RUN_SEED}

        npz_files = sorted(pathlib.Path(LOCAL_HELDOUT_CACHE).glob('*.npz'))
        print(f'\n{len(npz_files)} held-out .npz files (manifest written by Section 8c)')
        if len(npz_files) == 0:
            print('Preprocess returned 0 files - held-out skipped')
            HELDOUT_AVAILABLE = False
else:
    print('HELDOUT_AVAILABLE=False - skipping')

# produced no files, print a comprehensive diagnostic and bail. If
# HELDOUT_AVAILABLE is False (raw zips missing, etc.), this is a normal
# skip and we must NOT raise — the rest of the notebook treats held-out
# as best-effort (see Section 8 header and cell 70's guard).
import pathlib as _p
_local = _p.Path(LOCAL_HELDOUT_CACHE)
_npz_count = len(list(_local.glob('*.npz'))) if _local.exists() else 0
if HELDOUT_AVAILABLE and _npz_count == 0:
    _drive_c = _p.Path(DRIVE_PREPROCESSED_1500)
    _drive_z = _p.Path(DRIVE_ZIPS)
    print('='*60)
    print('CACHE DIAGNOSTIC (held-out cache is empty)')
    print('='*60)
    print(f'LOCAL_HELDOUT_CACHE     = {LOCAL_HELDOUT_CACHE}')
    print(f'  exists                = {_local.exists()}')
    print(f'  .npz count            = {_npz_count}')
    print(f'DRIVE_PREPROCESSED_1500 = {DRIVE_PREPROCESSED_1500}')
    print(f'  exists                = {_drive_c.exists()}')
    if _drive_c.exists():
        print(f'  .npz count            = {len(list(_drive_c.glob("*.npz")))}')
    print(f'DRIVE_ZIPS              = {DRIVE_ZIPS}')
    print(f'  exists                = {_drive_z.exists()}')
    if _drive_z.exists():
        zips = sorted(_drive_z.glob('*.zip'))
        print(f'  .zip files            = {[z.name for z in zips]}')
    print(f'LOCAL_LOGS              = {LOCAL_LOGS}')
    _logs_p = _p.Path(LOCAL_LOGS)
    if _logs_p.exists():
        print(f'  .db count             = {len(list(_logs_p.glob("*.db")))}')
        nested_dbs = list(_logs_p.rglob('*.db'))
        if nested_dbs and not any(_logs_p.glob('*.db')):
            print(f'  NESTED .db files found at: {nested_dbs[0].parent}')
            print(f'  -> auto-flatten cell did not run or did not catch this layout')
    print(f'LOCAL_MAPS              = {LOCAL_MAPS}')
    _maps_p = _p.Path(LOCAL_MAPS)
    if _maps_p.exists():
        print(f'  has nuplan-maps-v1.0.json: {(_maps_p/"nuplan-maps-v1.0.json").exists()}')
        nested = list(_maps_p.rglob('nuplan-maps-v1.0.json'))
        if nested and not (_maps_p/"nuplan-maps-v1.0.json").exists():
            print(f'  NESTED maps json found at: {nested[0]}')
    print('='*60)
    print('Likely fixes:')
    print('  - If Drive zips are missing: upload nuplan-maps-v1.0.zip and')
    print('    nuplan-v1.1_mini.zip to', DRIVE_ZIPS)
    print('  - If files are at nested paths: re-run the auto-flatten cell')
    print('    that follows the extract cell, then re-run preprocess.')
    print('  - If you have a v6 Drive cache at a different path: update')
    print('    DRIVE_PREPROCESSED_1500 in the config cell.')
    print('='*60)
    raise RuntimeError(f'Held-out cache validation failed - 0 .npz files at {LOCAL_HELDOUT_CACHE}. See diagnostic above.')
elif _npz_count == 0:
    print(f'HELDOUT_AVAILABLE=False and no held-out cache at {LOCAL_HELDOUT_CACHE} '
          '— skipping diagnostic (this is the normal best-effort skip path).')

import os, subprocess, json, pathlib

if HELDOUT_AVAILABLE:
    r = subprocess.run(
        [PYTHON, f'{REPO_DIR}/scripts/validate_npz_cache.py'],
        capture_output=True, text=True,
        env={**os.environ,
             'CACHE_DIR':       LOCAL_HELDOUT_CACHE,
             'OUTPUT_MANIFEST': f'{LOCAL_HELDOUT_CACHE}/diffusion_planner_training.json',
             'FAIL_THRESHOLD':  '0.05'},
    )
    print(r.stdout)
    if r.returncode != 0:
        print('STDERR:'); print(r.stderr)
        print('\nHeld-out validation failed — disabling held-out eval downstream')
        HELDOUT_AVAILABLE = False
    else:
        # Defense in depth: ensure no filename overlap between train and held-out
        # caches. The log-disjoint split guarantees this by construction; the
        # check exists so any methodology regression that bypasses the split
        # is caught.
        manifest_path = pathlib.Path(f'{LOCAL_HELDOUT_CACHE}/diffusion_planner_training.json')
        heldout_names = set(json.loads(manifest_path.read_text()))
        train_names = set(p.name for p in pathlib.Path(LOCAL_CACHE).glob('*.npz'))
        overlap = heldout_names & train_names
        if overlap:
            print(f'\n!!! UNEXPECTED train/heldout overlap: {len(overlap)} files')
            print('!!! The log-disjoint split should guarantee disjoint sets.')
            print('!!! Investigate before trusting this held-out cache.')
            print('!!! Removing overlapping files from manifest.')
            heldout_names -= overlap
            manifest_path.write_text(json.dumps(sorted(heldout_names)))
        print(f'\nHeld-out ready: {len(heldout_names)} scenarios (log-disjoint from train)')
else:
    print('HELDOUT_AVAILABLE=False — skipping held-out validation')


---

## 5. Train + eval each of the 4 kinematics

Same architecture, same seed (269), same hyperparameters. The ONLY flag that
varies is `model.kinematic=`. No `+centerline_encoder` overrides. After each
training, copy ckpt to Drive and run eval.

In [ ]:
import os, json, subprocess, time, pathlib

DRIVE_CKPT_M    = f'{DRIVE_OUT_MOTION}/checkpoints'
DRIVE_RESULTS_M = f'{DRIVE_OUT_MOTION}/results'
DRIVE_GIFS_M    = f'{DRIVE_OUT_MOTION}/gifs'
DRIVE_FEAS_M    = f'{DRIVE_OUT_MOTION}/feasibility'
matrix_rows = []

NORM_STATS_BY_KIN = {
    'waypoints':    'waypoints_norm_stats',
    'velocity':     'waypoints_norm_stats',
    'acceleration': 'waypoints_norm_stats',
    'frenet':       'frenet_norm_stats',
}

def run_eval_motion(ckpt_path, label, kinematic):
    out_json = f'{DRIVE_RESULTS_M}/eval_{label}.json'
    cmd = [
        PYTHON, '-m', 'flow_planner.run_script.inference_eval',
        '--checkpoint',  ckpt_path,
        '--data_dir',    LOCAL_HELDOUT_CACHE,
        '--data_list',   f'{LOCAL_HELDOUT_CACHE}/diffusion_planner_training.json',
        '--kinematic',   kinematic,
        '--norm_stats',  NORM_STATS_BY_KIN[kinematic],
        '--output_json', out_json,
        '--batch_size',  str(EFFECTIVE_BATCH_SIZE),
        '--num_batches', '999',
        '--seed',        '0',
        '--no_centerline_encoder',
    ]
    %cd {FP_DIR}
    print(f'\n--- run_eval_motion[{label}] -> {out_json} ---')
    t0 = time.time()
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print('STDOUT (last 30):'); print('\n'.join(r.stdout.split('\n')[-30:]))
        print('STDERR (last 30):'); print('\n'.join(r.stderr.split('\n')[-30:]))
        raise RuntimeError(f'run_eval_motion[{label}] failed exit={r.returncode}')
    dt = time.time() - t0
    with open(out_json) as f:
        result = json.load(f)
    print(f'  done in {dt:.0f}s  ADE={result.get("ade_mean"):.2f}  FDE={result.get("fde_mean"):.2f}')
    row = {
        'variant': label, 'kinematic': kinematic, 'ckpt_path': ckpt_path,
        'ade_mean': result.get('ade_mean'), 'ade_std': result.get('ade_std'),
        'fde_mean': result.get('fde_mean'), 'fde_std': result.get('fde_std'),
        'n_scenarios': result.get('num_scenarios_evaluated'),
        'json_path': out_json,
    }
    matrix_rows.append(row)
    return result

### Training loop — 4 kinematics × 1 seed

In [ ]:
import os, pathlib, shutil, subprocess, json

os.environ['PROJECT_ROOT']         = FP_DIR
os.environ['SAVE_DIR']             = LOCAL_RUNS
os.environ['TENSORBOARD_LOG_PATH'] = LOCAL_TB
os.environ['TRAINING_DATA']        = LOCAL_CACHE
os.environ['TRAINING_JSON']        = f'{LOCAL_CACHE}/diffusion_planner_training.json'
os.environ['WORLD_SIZE']           = '1'
os.environ['LOCAL_RANK']           = '0'
os.environ['MASTER_ADDR']          = 'localhost'
os.environ['MASTER_PORT']          = '29529'
os.environ['HYDRA_FULL_ERROR']     = '1'
for k in ['FRENET_TANH_D', 'FRENET_TANH_D_SCALE', 'FRENET_SMART_CENTERLINE',
          'FRENET_INFERENCE_HOOK', 'FRENET_INFERENCE_K', 'FRENET_INFERENCE_N',
          'FRENET_INFERENCE_TEMPERATURE', 'FRENET_INFERENCE_VERIFIER',
          'FRENET_INFERENCE_VERIFIER_ALPHA']:
    os.environ.pop(k, None)

n_files = len(json.loads(pathlib.Path(os.environ['TRAINING_JSON']).read_text()))
print(f'Training manifest: {n_files} scenarios per kinematic.')

for seed in SEEDS_TO_RUN:
    for kinematic in KINEMATICS_TO_RUN:
        run_name = f'motion_{kinematic}_seed{seed}'
        print(f'\n{"="*70}\nTRAINING: {kinematic} seed={seed}\n{"="*70}')

        runs_dir = pathlib.Path(LOCAL_RUNS)
        if runs_dir.exists():
            shutil.rmtree(runs_dir)
        runs_dir.mkdir(parents=True, exist_ok=True)

        drive_ckpt = f'{DRIVE_CKPT_M}/{run_name}.ckpt'
        if pathlib.Path(drive_ckpt).exists():
            print(f'  ckpt exists at {drive_ckpt} — skipping training')
        else:
            %cd {FP_DIR}
            cmd = (
                f'{VENV}/bin/torchrun --nnodes 1 --nproc-per-node 1 --standalone '
                f'flow_planner/trainer.py --config-name flow_planner_standard '
                f'model.kinematic={kinematic} '
                f'normalization_stats={NORM_STATS_BY_KIN[kinematic]} '
                f'train.batch_size={EFFECTIVE_BATCH_SIZE} '
                f'train.epoch={TRAIN_EPOCHS} '
                f'scheduler.warm_up_epoch={WARM_UP_EPOCHS} '
                f'train.save_utd=1 '
                f'save_every_since={TRAIN_EPOCHS} '
                f'ddp.distributed=false '
                f'seed={seed} '
                f'job_name={run_name} '
                f'num_workers=2'
            )
            log_path = pathlib.Path(LOCAL_RUNS) / f'{run_name}.log'
            print(f'  log -> {log_path}')
            with open(log_path, 'w') as logf:
                r = subprocess.run(['bash', '-c', cmd], stdout=logf, stderr=subprocess.STDOUT)
            !tail -15 {log_path}
            assert r.returncode == 0, f'Training failed for {run_name}'

            pths = sorted(pathlib.Path(LOCAL_RUNS).rglob('latest.pth'),
                          key=lambda p: p.stat().st_mtime, reverse=True)
            assert pths, f'No latest.pth produced'
            shutil.copy(pths[0], drive_ckpt)
            print(f'  ckpt -> {drive_ckpt}')

        run_eval_motion(drive_ckpt, f'motion_{kinematic}_seed{seed}', kinematic)

print(f'\nAll {len(SEEDS_TO_RUN)*len(KINEMATICS_TO_RUN)} combos trained + evaluated.')

## 6. Bicycle-feasibility post-process analysis

For each variant's per-scene trajectories, compute kinematic-feasibility stats
against a bicycle motion model:
- max longitudinal acceleration  (bound 4 m/s² for comfort)
- max lateral acceleration  (bound 4 m/s²)
- max heading rate / implied steering angle  (bound 0.7 rad)

Output: per-variant fraction-of-trajectories-passing-all-bounds metric.

This answers a different question from ADE/FDE: "Is the predicted trajectory
something a car could actually drive?" A model with low ADE but lots of
infeasible trajectories is overfitting to numeric distance and ignoring the
kinematic structure of the task. A model with higher ADE but high feasibility
might be a better deployment candidate.

In [ ]:
if RUN_FEASIBILITY and matrix_rows:
    import json, numpy as np

    DT = 0.1  # nuPlan frame interval in seconds
    A_LON_MAX = 4.0
    A_LAT_MAX = 4.0
    STEER_MAX = 0.7  # rad

    feas_rows = []
    for row in matrix_rows:
        if not row.get('json_path'):
            continue
        with open(row['json_path']) as f:
            j = json.load(f)
        per_scene_pred = j.get('per_scene_pred_traj')
        if per_scene_pred is None:
            print(f'  {row["variant"]:<30} (per_scene_pred_traj not in JSON; skipping)')
            continue
        trajs = np.array(per_scene_pred)
        if trajs.ndim < 3:
            continue
        # Expect (N_scenes, T_frames, >=2 [x, y, ...])
        xy = trajs[..., :2]
        # Velocity = (xy[t+1] - xy[t]) / DT, shape (N, T-1, 2)
        vel = (xy[:, 1:, :] - xy[:, :-1, :]) / DT
        speed = np.linalg.norm(vel, axis=-1)
        # Acceleration = (vel[t+1] - vel[t]) / DT, shape (N, T-2, 2)
        acc = (vel[:, 1:, :] - vel[:, :-1, :]) / DT
        # Heading from velocity direction; defined where speed > eps.
        eps = 1e-3
        heading = np.arctan2(vel[..., 1], vel[..., 0])
        heading_rate = np.zeros_like(heading[:, :-1])
        heading_rate[:, :] = np.diff(np.unwrap(heading, axis=-1), axis=-1) / DT
        # Decompose accel into longitudinal/lateral using heading direction
        cos_h = np.cos(heading[:, :-1])
        sin_h = np.sin(heading[:, :-1])
        a_lon = acc[..., 0] * cos_h + acc[..., 1] * sin_h
        a_lat = -acc[..., 0] * sin_h + acc[..., 1] * cos_h
        # Implied steering angle from heading_rate * wheelbase / speed
        WHEELBASE = 2.8
        safe_speed = np.where(speed[:, :-1] > eps, speed[:, :-1], np.nan)
        steer = np.arctan2(heading_rate * WHEELBASE, safe_speed)

        max_a_lon_per_scene = np.nanmax(np.abs(a_lon), axis=-1)
        max_a_lat_per_scene = np.nanmax(np.abs(a_lat), axis=-1)
        max_steer_per_scene = np.nanmax(np.abs(steer), axis=-1)

        feasible_mask = (
            (max_a_lon_per_scene <= A_LON_MAX)
            & (max_a_lat_per_scene <= A_LAT_MAX)
            & (max_steer_per_scene <= STEER_MAX)
        )
        feas_row = {
            'variant': row['variant'],
            'kinematic': row['kinematic'],
            'frac_feasible': float(np.mean(feasible_mask)),
            'mean_max_a_lon': float(np.nanmean(max_a_lon_per_scene)),
            'mean_max_a_lat': float(np.nanmean(max_a_lat_per_scene)),
            'mean_max_steer_rad': float(np.nanmean(max_steer_per_scene)),
            'n_scenes': int(len(trajs)),
        }
        feas_rows.append(feas_row)
        print(f'  {row["variant"]:<30}  '
              f'frac_feasible={feas_row["frac_feasible"]:.3f}  '
              f'a_lon={feas_row["mean_max_a_lon"]:.2f}  '
              f'a_lat={feas_row["mean_max_a_lat"]:.2f}  '
              f'steer={feas_row["mean_max_steer_rad"]:.3f}')

    if feas_rows:
        import pandas as pd
        feas_df = pd.DataFrame(feas_rows).sort_values('frac_feasible', ascending=False)
        feas_csv = f'{DRIVE_FEAS_M}/bicycle_feasibility.csv'
        feas_df.to_csv(feas_csv, index=False)
        print(f'\nwrote {feas_csv}')
        print(feas_df.to_string(index=False))
else:
    print('Feasibility analysis skipped or no matrix rows.')

## 7. Comparison table + bar chart

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

if matrix_rows:
    df = pd.DataFrame(matrix_rows).sort_values('ade_mean').reset_index(drop=True)
    out_csv = f'{DRIVE_RESULTS_M}/motion_matrix.csv'
    df.to_csv(out_csv, index=False)
    print(f'wrote {out_csv}\n')
    print(df[['variant','kinematic','ade_mean','ade_std','fde_mean','fde_std','n_scenarios']].to_string(index=False))

    fig, ax = plt.subplots(figsize=(10, 5))
    colors = {'waypoints':'steelblue','velocity':'mediumseagreen',
              'acceleration':'goldenrod','frenet':'tomato'}
    bars = [colors.get(r['kinematic'],'gray') for _,r in df.iterrows()]
    ax.bar(df['variant'], df['ade_mean'], yerr=df['ade_std'], capsize=4, color=bars, alpha=0.85)
    ax.set_ylabel('Heldout ADE (m)')
    ax.set_title('Motion-rep zoo (4 kinematics, unmodified Flow Planner)')
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
    ax.grid(alpha=0.3, axis='y')
    handles = [plt.Rectangle((0,0),1,1, color=c) for k,c in colors.items() if k in df['kinematic'].values]
    labels  = [k for k in colors if k in df['kinematic'].values]
    ax.legend(handles, labels, title='Kinematic')
    fig.tight_layout()
    plt.savefig(f'{DRIVE_RESULTS_M}/motion_matrix.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print('No matrix_rows.')

## 8. Rollout gifs (4 fixed heldout scenes per variant)

In [ ]:
import os, pathlib, subprocess

for row in matrix_rows:
    variant = row['variant']
    kin = row['kinematic']
    gif_dir = f'{DRIVE_GIFS_M}/{variant}'
    pathlib.Path(gif_dir).mkdir(parents=True, exist_ok=True)
    env = dict(os.environ)
    env.update({
        'FP_DIR': FP_DIR, 'CKPT_PATH': row['ckpt_path'], 'CACHE_DIR': LOCAL_HELDOUT_CACHE,
        'OUTPUT_DIR': gif_dir, 'KINEMATIC': kin, 'PLOT_TITLE': variant,
        'SCENE_INDICES': '57,12,140,125', 'GIF_FPS': '10',
    })
    cmd = [PYTHON, f'{REPO_DIR}/scripts/visualize_bev_gif.py']
    print(f'\n--- gifs[{variant}] -> {gif_dir} ---')
    r = subprocess.run(cmd, env=env, capture_output=True, text=True)
    if r.returncode != 0:
        print('STDERR (last 20):'); print('\n'.join(r.stderr.split('\n')[-20:]))
        print(f'  (gif render failed for {variant}; continuing)')
    else:
        print(r.stdout[-400:])

## 9. Motion-rep zoo report card

In [ ]:
if matrix_rows:
    df = pd.DataFrame(matrix_rows).sort_values('ade_mean').reset_index(drop=True)
    winner = df.iloc[0]
    feas_summary = ''
    feas_csv_path = f'{DRIVE_FEAS_M}/bicycle_feasibility.csv'
    if pathlib.Path(feas_csv_path).exists():
        feas_df = pd.read_csv(feas_csv_path)
        feas_summary = '\n\n**Bicycle-feasibility rates:**\n\n' + feas_df.to_string(index=False)

    report = f'''# Motion-Representation Zoo — Report

**Setup:** unmodified Flow Planner (no Option A, no audit overrides),
{TRAIN_EPOCHS} epochs, batch {EFFECTIVE_BATCH_SIZE}, seed {SEEDS_TO_RUN[0]},
1500 train scenarios, 300 heldout.

**ADE ranking:**

{df[['variant','kinematic','ade_mean','ade_std','fde_mean','fde_std']].to_string(index=False)}

**Best:** `{winner["variant"]}` — ADE={winner["ade_mean"]:.2f} ± {winner["ade_std"]:.2f}
{feas_summary}

**Artifacts on Drive ({DRIVE_OUT_MOTION}):**
- `results/motion_matrix.csv`, `results/motion_matrix.png`
- `feasibility/bicycle_feasibility.csv`
- `gifs/<variant>/rollout_*.gif`
- `checkpoints/*.ckpt`
'''
    print(report)
    with open(f'{DRIVE_OUT_MOTION}/REPORT_CARD.md', 'w') as f:
        f.write(report)
    print(f'\nwrote {DRIVE_OUT_MOTION}/REPORT_CARD.md')